# 04 — Manual Agent Loop, `create_agent` & Streaming

## Learning requirements
- tự implement một agent loop trước khi dùng abstraction;
- dùng `create_agent` của LangChain v1;
- inspect agent state/messages;
- stream agent progress/tokens;
- đặt max-iteration/budget mindset.

`create_agent` là high-level agent API hiện tại. `langgraph.prebuilt.create_react_agent` không phải learning path chính cho project mới.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
while not (root / "requirements.txt").exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

from src.providers import get_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

model = get_chat_model()
model_with_tools = model.bind_tools([multiply])

messages = [HumanMessage("What is 19 * 23? Use the tool.")]
for step in range(5):
    ai = model_with_tools.invoke(messages)
    messages.append(ai)

    if not ai.tool_calls:
        print(ai.text if hasattr(ai, "text") else ai.content)
        break

    for call in ai.tool_calls:
        if call["name"] != "multiply":
            raise RuntimeError(f"Unknown tool: {call['name']}")
        result = multiply.invoke(call["args"])
        messages.append(ToolMessage(
            content=str(result),
            tool_call_id=call["id"],
            name=call["name"],
        ))
else:
    raise RuntimeError("Agent exceeded max steps")

## Chuyển sang `create_agent`

Khi đã hiểu loop, LangChain xử lý orchestration/tool loop giúp ta, và LangGraph runtime nằm bên dưới agent.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=get_chat_model(),
    tools=[multiply],
    system_prompt="You are a careful assistant. Use tools when arithmetic is required.",
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Calculate 19 * 23 and explain in one sentence."}]
})

for m in result["messages"]:
    print(type(m).__name__, "=>", getattr(m, "content", None))

In [ ]:
# Streaming: inspect update events.
for event in agent.stream(
    {"messages": [{"role": "user", "content": "What is 123 * 17?"}]},
    stream_mode="updates",
):
    print(event)

## Exercise

Build `research_agent_v1` với 3 tools:
- calculator,
- local document lookup,
- project metadata lookup.

Bổ sung:
- structured final result;
- streaming;
- clear failure messages;
- iteration/token budget note.

## Required output
Agent phải cho thấy trong log/message history:
`Human -> AI(tool_call) -> ToolMessage -> AI(final)`.

## Done criteria
Bạn giải thích được phần nào do model quyết định và phần nào do runtime/application quyết định.